#### Load Settings

In [1]:
from llama_index.core import Settings
Settings.llm = None
Settings.chunk_size = 256
Settings.chunk_overlap = 25

LLM is explicitly disabled. Using MockLLM.


### Load nodes from disk


#### load storage context

In [2]:
from llama_index.core import Settings, SimpleKeywordTableIndex, StorageContext

In [3]:
storage_context = StorageContext.from_defaults(
    persist_dir="/Users/mattan/Library/CloudStorage/GoogleDrive-mattany@gmail.com/My Drive/RAG/content/Nodes")

#### build keyword index

In [4]:
from llama_index.core import SimpleKeywordTableIndex

nodes = list(storage_context.docstore.docs.values())


class QuickerSimpleKeywordTableIndex(SimpleKeywordTableIndex):
    def build_index_from_nodes(self, nodes):
        """Build the index from nodes."""
        return self._build_index_from_nodes(nodes)


keyword_index = QuickerSimpleKeywordTableIndex(nodes=nodes, storage_context=storage_context, show_progress=True)

Extracting keywords from nodes:   0%|          | 0/298529 [00:00<?, ?it/s]


#### define query engine

In [5]:
from llama_index.core import PromptTemplate
from llama_index.core.retrievers import (
    KeywordTableSimpleRetriever,
)
from llama_index.core.query_engine import RetrieverQueryEngine

In [26]:

qa_prompt_tmpl_str = (
"""<CONTEXT>
{context_str}
</CONTEXT>
 Given the context above, answer the following question:
 Question:
 With regards to the solar system, {query_str}
 Answer:
 """
)
qa_prompt_tmpl = PromptTemplate(qa_prompt_tmpl_str)


keyword_retriever = KeywordTableSimpleRetriever(index=keyword_index)

query_engine = RetrieverQueryEngine(
    retriever=keyword_retriever,
)
query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)


### Load model

In [19]:
!pip install mlx-lm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip


In [20]:
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/Meta-Llama-3-8B-4bit")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

#### get response from query engine

In [43]:

# query_engine_query = "Titan (moon)"
query = "Is mars hotter or colder than the earth?"

no_context_prompt = f"""answer the following question:
 Question:
 With regards to the solar system, {query}
 Answer:"""

query_engine_response = query_engine.query(
    query
)
print(query_engine_response)

<CONTEXT>
id: 9596342
url: https://en.wikipedia.org/wiki/Climate%20of%20Mars
title: Climate of Mars

Before and after the Viking missions, newer, more advanced Martian temperatures were determined from Earth via microwave spectroscopy.  As the microwave beam, of under 1 arcminute, is larger than the disk of the planet, the results are global averages.  Later, the Mars Global Surveyor's Thermal Emission Spectrometer and to a lesser extent 2001 Mars Odyssey's THEMIS could not merely reproduce infrared measurements but intercompare lander, rover, and Earth microwave data.  The Mars Reconnaissance Orbiter's Mars Climate Sounder can similarly derive atmospheric profiles. The datasets "suggest generally colder atmospheric temperatures and lower dust loading in recent decades on Mars than during the Viking Mission," although Viking data had previously been revised downward.  The TES data indicates "Much colder (10–20 K) global atmospheric temperatures were observed during the 1997 versus 1977

#### model response without context

In [44]:
zero_shot_response = generate(model, tokenizer, prompt=no_context_prompt, verbose=True, max_tokens=256, temp=0)
print(zero_shot_response)

Prompt: answer the following question:
 Question:
 With regards to the solar system, Is mars hotter or colder than the earth?
 Answer:
 Mars is colder than earth. Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy. Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Solution:
Mars is colder than earth. Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Mars is a planet with a very thin atmosphere. It is very cold and dry. It is also very windy.
Mars is a planet with a very thin atmosphere. It

#### model response with context

In [45]:
rag_response = generate(model, tokenizer, prompt=query_engine_response.response, verbose=True, max_tokens=256, temp=0.1)
print(rag_response)

Prompt: <CONTEXT>
id: 9596342
url: https://en.wikipedia.org/wiki/Climate%20of%20Mars
title: Climate of Mars

Before and after the Viking missions, newer, more advanced Martian temperatures were determined from Earth via microwave spectroscopy.  As the microwave beam, of under 1 arcminute, is larger than the disk of the planet, the results are global averages.  Later, the Mars Global Surveyor's Thermal Emission Spectrometer and to a lesser extent 2001 Mars Odyssey's THEMIS could not merely reproduce infrared measurements but intercompare lander, rover, and Earth microwave data.  The Mars Reconnaissance Orbiter's Mars Climate Sounder can similarly derive atmospheric profiles. The datasets "suggest generally colder atmospheric temperatures and lower dust loading in recent decades on Mars than during the Viking Mission," although Viking data had previously been revised downward.  The TES data indicates "Much colder (10–20 K) global atmospheric temperatures were observed during the 1997 ver